# 01 — Preprocessing and Quality Control

Loads the raw Roux et al. (2022) MSC screen (GEO GSE176206), reduces the object
to its counts matrix to manage memory, applies quality-control filtering,
detects and removes doublets, and writes a cleaned checkpoint
(`msc_clean_raw.h5ad`) used by all downstream notebooks.

In [ ]:
import sys
!{sys.executable} -m pip install "numpy==1.26.4" scanpy anndata scikit-image --quiet

from google.colab import drive
drive.mount('/content/drive')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 58.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.0/244.0 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 76.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have nu

In [ ]:
# copy the already-downloaded file from Drive to local disk, then unzip
!cp "/content/drive/MyDrive/roux_project/msc_screen.h5ad.gz" /content/msc_screen.h5ad.gz
!gunzip -f /content/msc_screen.h5ad.gz
!ls -lh /content/msc_screen.h5ad


-rw------- 1 root root 4.7G Sep  3 15:11 /content/msc_screen.h5ad


In [ ]:
import anndata as ad
import scanpy as sc

adata = ad.read_h5ad("/content/msc_screen.h5ad")
adata.X = adata.layers['counts'].copy()   # keep raw counts as the working matrix
del adata.layers['log1p_cpm']             # drop unneeded normalised layers
del adata.layers['scvi_normalized']
adata.raw = None                          # drop the large (28,701-gene) .raw
import gc; gc.collect()

print(adata.shape)
print("RAM check:")
!free -h


(10021, 19321)
RAM check:
               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.1Gi       2.5Gi       3.0Mi       8.1Gi        10Gi
Swap:             0B          0B          0B


In [ ]:
print("X min:", adata.X.min())
print("X max:", adata.X.max())
import numpy as np
print("all integers?", np.allclose(adata.X.data, np.round(adata.X.data)))


X min: 0.0
X max: 27181.0
all integers? True


In [ ]:
# quality control: keep cells with >=500 genes detected and <10% mitochondrial content
adata = adata[
    (adata.obs['n_genes'] >= 500) &
    (adata.obs['perc_mito'] < 0.10)
].copy()
print(adata.shape)

(10021, 19321)


In [ ]:
# doublet detection on the HVG subset to limit memory, then transfer labels back
adata_hvg = adata.copy()
sc.pp.normalize_total(adata_hvg, target_sum=1e4)
sc.pp.log1p(adata_hvg)
sc.pp.highly_variable_genes(adata_hvg, n_top_genes=3000)
adata_hvg = adata_hvg[:, adata_hvg.var['highly_variable']].copy()
sc.pp.scrublet(adata_hvg, random_state=0)
adata.obs['predicted_doublet'] = adata_hvg.obs['predicted_doublet']
adata.obs['doublet_score'] = adata_hvg.obs['doublet_score']
del adata_hvg
import gc; gc.collect()
print(adata.obs['predicted_doublet'].value_counts())

predicted_doublet
False    10018
True         3
Name: count, dtype: int64


In [ ]:
adata = adata[~adata.obs['predicted_doublet']].copy()
print(adata.shape)

(10018, 19321)


In [ ]:
adata.layers['raw_count'] = adata.X.copy()
adata.write_h5ad("/content/drive/MyDrive/roux_project/msc_clean_raw.h5ad")
print("saved — shape:", adata.shape)

saved — shape: (10018, 19321)


## Output

Cleaned dataset: 10,018 cells × 19,321 genes (3 doublets removed; QC thresholds
removed no additional cells, as the deposited data was pre-filtered). The
`raw_count` layer preserves the raw counts for downstream normalisation. Saved
as `msc_clean_raw.h5ad`.